# Lecture 2: Building a Neural Network — Plain Language Guide

This notebook explains every concept from the neural network exercise in simple, everyday language — no maths degree required.

**The business scenario throughout this notebook:**
A retail company wants to predict which website visitors are likely to make a purchase — so they can personalise offers in real time and increase conversion rates.

---

**What we cover:**
1. What a neural network actually is
2. The dataset — what we are working with
3. Preparing data for AI
4. Turning data into the format the model needs
5. Building the neural network
6. The learning mechanism — loss and optimiser
7. The training loop — how the model actually learns
8. Reading the results — loss curves
9. Evaluating performance — beyond simple accuracy


## Section 1: What Is a Neural Network?

---

### The One-Sentence Version

A neural network is a system that learns to make predictions by repeatedly adjusting thousands of internal dials — until its predictions are as accurate as possible.

---

### The Business Analogy

Imagine you are training a new analyst to predict which customers will buy. On day one, the analyst knows nothing, so they make random guesses. After each prediction, you tell them whether they were right or wrong, and they adjust their thinking slightly.

After seeing thousands of examples, the analyst has developed an internal sense of which patterns lead to a purchase — even for customers they have never seen before. They cannot fully explain their reasoning, but their predictions are accurate.

A neural network works the same way. It starts with random guesses, gets feedback on how wrong it was, adjusts its internal settings, and repeats — millions of times.

---

### How It Is Structured

A neural network is organised in layers:

| Layer | Role | Business analogy |
|-------|------|-----------------|
| **Input layer** | Receives the raw data | The raw customer data arriving on your desk |
| **Hidden layers** | Find patterns and relationships in the data | The analyst's internal reasoning process |
| **Output layer** | Produces the final prediction | The analyst's final answer: buy or not buy |

The "hidden" layers are where the learning happens. You cannot directly see what they are doing — but they are building increasingly abstract representations of the patterns in your data.

---

### What Makes It "Deep"?

A neural network with multiple hidden layers is called a **deep neural network** — hence "deep learning." More layers allow the network to learn more complex patterns. The exercise in this lecture uses two hidden layers, which is sufficient for most tabular business prediction tasks.


## Section 2: The Dataset — Online Shoppers Purchasing Intention

---

### What Are We Trying to Predict?

The goal is **binary classification**: for each website session, predict whether it ended in a purchase (`True`) or not (`False`). This is a yes/no question — the most common type of business prediction problem.

Examples of similar business questions:
- Will this loan applicant default? (yes/no)
- Will this customer churn next month? (yes/no)
- Is this transaction fraudulent? (yes/no)

---

### What Data Do We Have?

The dataset contains 12,330 rows — one per website session — collected from a real e-commerce site. For each session, we know:

| Category | What it captures |
|----------|-----------------|
| Page visits | How many administrative, informational, and product pages the user visited |
| Time on pages | How long they spent on each type of page |
| Google Analytics signals | Bounce rate, exit rate, page value, proximity to special days |
| Session context | Month, operating system, browser, region, traffic source |
| Visitor profile | New visitor, returning visitor, or other |
| Timing | Whether the session occurred on a weekend |

**Target:** Did this session result in a purchase?

---

### The Imbalance Problem

Roughly **84% of sessions do not result in a purchase**. Only 16% do.

This is called a **class imbalance**, and it is extremely common in business datasets:
- Most loan applications are not fraudulent
- Most customers do not churn
- Most website visitors do not buy

This matters because a model that simply predicts "no purchase" every single time would be 84% accurate — but completely useless. We need metrics that reveal whether the model is genuinely learning to identify buyers, not just defaulting to the majority class.


## Section 3: Preparing Data — Why Raw Data Cannot Go Straight Into a Model

---

### The Core Problem

Neural networks only understand numbers. But real-world data contains:
- Text categories (e.g. "November", "Returning Visitor")
- Boolean flags (True/False)
- Numbers on wildly different scales (e.g. page visit count: 0–100 vs. time on page: 0–10,000 seconds)

Before training can begin, all of this must be standardised.

---

### Step 1: Encoding Categories — Turning Words Into Numbers

The dataset has columns like `Month` ("Jan", "Feb", …, "Dec") and `VisitorType` ("Returning Visitor", "New Visitor"). These cannot be fed to a neural network as text.

The solution is **one-hot encoding**: each possible category becomes its own column, with a 1 if that category applies and 0 otherwise.

**Example — Month column:**

| Original | Jan | Feb | Mar | … |
|----------|-----|-----|-----|---|
| "Jan"    | 1   | 0   | 0   | … |
| "Mar"    | 0   | 0   | 1   | … |

> **Why not just assign numbers?** Encoding "Jan=1, Feb=2, Mar=3" implies March is three times January, which is meaningless. One-hot encoding treats each category independently.

---

### Step 2: Splitting Into Training and Test Sets

The data is split into two parts:

- **Training set (80%)** — the model learns from this data
- **Test set (20%)** — the model is evaluated on data it has never seen

The test set acts as the final exam. A model that scores well only on training data has simply memorised the answers — like a student who memorises past exam papers but cannot handle new questions. This is called **overfitting**.

The split uses **stratification** — ensuring both sets have the same proportion of buyers and non-buyers. Without this, random chance could put most of the rare purchase cases into one set, making evaluation misleading.

---

### Step 3: Normalisation — Putting All Features on the Same Scale

Some features have values in the range 0–1. Others are in the range 0–50,000. If left as-is, features with large values would dominate the model's learning simply due to their scale — not because they are more important.

Normalisation rescales all features so they have a mean of 0 and a standard deviation of 1. Every feature is now on a level playing field.

> **Critical rule:** The normalisation formula (mean and standard deviation) is calculated using **training data only** and then applied to the test set. Using test data to compute the formula would be like showing a student the exam answers before marking them — it artificially inflates performance.


## Section 4: DataLoaders — Feeding Data to the Model Efficiently

---

### Why Not Just Pass All the Data at Once?

With 12,330 rows, passing everything to the model simultaneously would require enormous memory — and would actually produce a less stable model.

Instead, the data is divided into **batches** of 64 rows at a time. The model processes one batch, updates its internal settings, processes the next batch, updates again, and so on.

---

### One Pass Through All Batches = One Epoch

When the model has processed every batch in the training set once, that is called one **epoch**. In this exercise, we train for 30 epochs — meaning the model sees the entire dataset 30 times in total.

> **Analogy:** Imagine revising for an exam. Reading your notes once is one pass. Reading them 30 times — each time reinforcing and refining your understanding — is 30 epochs. Each pass improves retention.

---

### Training vs. Test DataLoader

Two separate data pipelines are created:

- **Training DataLoader** — shuffles the data each epoch (prevents the model from memorising the order)
- **Test DataLoader** — no shuffling (order does not matter for evaluation, and consistency makes debugging easier)

This separation ensures the model is always evaluated on data it has not trained on — the integrity of the evaluation depends on this.
